In [ ]:
import yaml
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F

ENV = 'DEV'  # 'DEV' | 'PRD'

session = get_active_session()

# อ่าน yml จาก Snowflake Stage
stage   = f'@DEMO_{ENV}.RAW.{ENV}_STAGE'
content = session.file.get_stream(f'{stage}/dev.yml').read().decode('utf-8')
cfg     = yaml.safe_load(content)

DB        = cfg['database']
ROLE      = cfg['snowflake']['role']
WAREHOUSE = cfg['warehouse']
SRC_TABLE = f'{DB}.RAW.SUPERSTORE_RAW'
TGT_TABLE = f'{DB}.SILVER.SALES_BY_PROVINCE'

print(f'ENV       : {ENV}')
print(f'DATABASE  : {DB}')
print(f'ROLE      : {ROLE}')
print(f'WAREHOUSE : {WAREHOUSE}')
print(f'SOURCE    : {SRC_TABLE}')
print(f'TARGET    : {TGT_TABLE}')

In [ ]:
# 2. Session
session = get_active_session()
session.sql(f'USE ROLE {ROLE}').collect()
session.sql(f'USE WAREHOUSE {WAREHOUSE}').collect()
session.sql(f'USE DATABASE {DB}').collect()
print('✅ Session ready')

In [ ]:
# 3. Extract
df_raw = session.table(SRC_TABLE)
print(f'RAW rows: {df_raw.count():,}')
df_raw.show(5)

In [ ]:
# 4. Transform - TEST VERSION
df_silver = (
    df_raw
    .filter(
        (F.col('SALES') > 0) &
        (F.col('PROVINCE').isNotNull())
    )
    .group_by('PROVINCE')
    .agg(
        F.round(F.sum('SALES'),  2).alias('TOTAL_SALES'),
        F.round(F.sum('PROFIT'), 2).alias('TOTAL_PROFIT'),
        F.round(F.avg('SALES'),  2).alias('AVG_SALES_PER_ORDER'),
        F.count('ORDER_ID').alias('TOTAL_ORDERS'),
        F.count_distinct('CUSTOMER_NAME').alias('UNIQUE_CUSTOMERS'),
    )
    .with_column('TEST_COL_1', F.lit('HELLO Aphisit Chumphon'))  # ← ค่าตายตัว ง่ายสุด
)
df_silver.show(10)

In [ ]:
# 5. Load → SILVER
session.sql(f'DROP TABLE IF EXISTS {TGT_TABLE}').collect()
df_silver.write.mode('overwrite').save_as_table(TGT_TABLE)
print(f'✅ Load สำเร็จ → {TGT_TABLE}')
print(f'   Rows: {session.table(TGT_TABLE).count():,}')
print(f'   Columns: {len(df_silver.columns)}')